In [1]:

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/uditjain13/heart-disease-risk-2026/heart_disease_risk_2026.csv


# Import libraries

In [2]:
import numpy as np
import pandas as pd

In [3]:
df = pd.read_csv('/kaggle/input/datasets/uditjain13/heart-disease-risk-2026/heart_disease_risk_2026.csv')

# General Understanding about dataset

In [ ]:
df.shape
df.info()
df.describe()
df.isnull().sum()
df.nunique()
df.duplicated().sum()
df.head()

# Preprocessing and cleaning (if needed)

In [ ]:
# drop exact duplicate rows if any
df = df.drop_duplicates().reset_index(drop=True)

# try to auto-detect the target column (common names for this kind of dataset)
possible_targets = ['target', 'risk', 'heart_disease', 'heart_disease_risk', 'disease',
                     'HeartDisease', 'HeartDiseaseRisk', 'risk_level', 'label', 'Risk']
target_col = None
for c in possible_targets:
    if c in df.columns:
        target_col = c
        break
if target_col is None:
    # fallback: assume last column is the target
    target_col = df.columns[-1]
print('Target column detected as:', target_col)

# fill missing numeric values with median, categorical with mode
for c in df.columns:
    if df[c].isnull().sum() > 0:
        if df[c].dtype == 'object':
            df[c] = df[c].fillna(df[c].mode()[0])
        else:
            df[c] = df[c].fillna(df[c].median())

# encode all remaining categorical (object) columns except the target
from sklearn.preprocessing import LabelEncoder

cat_cols = [c for c in df.select_dtypes(include='object').columns if c != target_col]
encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    df[c] = le.fit_transform(df[c].astype(str))
    encoders[c] = le

# encode the target if it's categorical (e.g. Yes/No, Low/Medium/High)
if df[target_col].dtype == 'object':
    le_target = LabelEncoder()
    df[target_col] = le_target.fit_transform(df[target_col].astype(str))
    print('Target classes:', list(le_target.classes_))

df.info()

# Feature Engineering

In [ ]:
# generic engineered features using whatever numeric columns exist
num_cols = [c for c in df.select_dtypes(include=np.number).columns if c != target_col]

# 1) average of all numeric risk indicators (a simple composite score)
if len(num_cols) > 1:
    df['risk_score_mean'] = df[num_cols].mean(axis=1)

# 2) age-based ratio feature, if an age-like column exists
age_col = next((c for c in df.columns if 'age' in c.lower()), None)
if age_col:
    for c in num_cols:
        if c != age_col:
            df[f'{c}_per_age'] = df[c] / (df[age_col] + 1)
            break  # just one example ratio to keep things simple

df.info()

# Model Training

In [ ]:
X = df.drop(target_col, axis=1)
y = df[target_col]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train, y_train)

model_dt = DecisionTreeClassifier(random_state=42)
model_dt.fit(X_train, y_train)

model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train, y_train)

print('Models trained: Logistic Regression, Decision Tree, Random Forest')

# Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

models = {
    'Logistic Regression': model_lr,
    'Decision Tree': model_dt,
    'Random Forest': model_rf
}

results = {}
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f'--- {name} ---')
    print('Accuracy:', acc)
    print(classification_report(y_test, y_pred))
    print('Confusion Matrix:')
    print(confusion_matrix(y_test, y_pred))
    print()

best_model_name = max(results, key=results.get)
print('Best model:', best_model_name, 'with accuracy', results[best_model_name])

# Hyperparameter tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=kf,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV accuracy:', grid_search.best_score_)

final_model = grid_search.best_estimator_
y_pred_final = final_model.predict(X_test)
print('Test accuracy after tuning:', accuracy_score(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

# Model dump

In [ ]:
import pickle
import joblib

with open('heart_disease_risk_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

joblib.dump(final_model, 'heart_disease_risk_model.joblib')

print('Model saved as heart_disease_risk_model.pkl and heart_disease_risk_model.joblib')